In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.autograd import Variable
import torch.optim as optim
import shap
import numpy as np
import pandas as pd
from IPython.display import clear_output
import matplotlib.pyplot as plt
import tqdm
import uproot

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
from sklearn.metrics import balanced_accuracy_score
import seaborn as sn
from tqdm import tqdm
import ROOT

%matplotlib inline

print(torch.__version__)


In [ ]:
def get_arrays(tree, branch_list):
    _dict = {}
    for _br in branch_list:
        _dict[_br] = getattr(tree[_br].arrays(), _br)
    return pd.DataFrame.from_dict(_dict)

def get_input_features(df, train_list, cuts=''):
    if cuts=='': return df[train_list].to_numpy()
    _df = df[df.eval(cuts)]
    return _df[train_list].to_numpy()

In [ ]:
def distance_corr(
        var_1:torch.tensor,
        var_2:torch.tensor,
        normedweight:torch.tensor,
        power=1,
        )->torch.tensor:
    """
    Compute the distance correlation function
    between two variables.

    Args:
        var_1 (torch.tensor): The first variable.
        var_2 (torch.tensor): The second variable.
        normedweight (torch.tensor): The weight matrix.
        power (int): The power of the distance correlation.

    Returns:
        torch.tensor: The distance correlation between the two variables.
    """
    
    # Normalize the weights
    normedweight = normedweight/torch.sum(normedweight)*len(var_1)
    
    xx = var_1.view(-1, 1).repeat(1, len(var_1)).view(len(var_1),len(var_1))
    yy = var_1.repeat(len(var_1),1).view(len(var_1),len(var_1))
    amat = (xx-yy).abs()

    xx = var_2.view(-1, 1).repeat(1, len(var_2)).view(len(var_2),len(var_2))
    yy = var_2.repeat(len(var_2),1).view(len(var_2),len(var_2))
    bmat = (xx-yy).abs()

    amatavg = torch.mean(amat*normedweight,dim=1)
    Amat=amat-amatavg.repeat(len(var_1),1).view(len(var_1),len(var_1))\
        -amatavg.view(-1, 1).repeat(1, len(var_1)).view(len(var_1),len(var_1))\
        +torch.mean(amatavg*normedweight)

    bmatavg = torch.mean(bmat*normedweight,dim=1)
    Bmat=bmat-bmatavg.repeat(len(var_2),1).view(len(var_2),len(var_2))\
        -bmatavg.view(-1, 1).repeat(1, len(var_2)).view(len(var_2),len(var_2))\
        +torch.mean(bmatavg*normedweight)

    ABavg = torch.mean(Amat*Bmat*normedweight,dim=1)
    AAavg = torch.mean(Amat*Amat*normedweight,dim=1)
    BBavg = torch.mean(Bmat*Bmat*normedweight,dim=1)

    if(power==1):
        dCorr=(torch.mean(ABavg*normedweight))/torch.sqrt((torch.mean(AAavg*normedweight)*torch.mean(BBavg*normedweight)))
    elif(power==2):
        dCorr=(torch.mean(ABavg*normedweight))**2/(torch.mean(AAavg*normedweight)*torch.mean(BBavg*normedweight))
    else:
        dCorr=((torch.mean(ABavg*normedweight))/torch.sqrt((torch.mean(AAavg*normedweight)*torch.mean(BBavg*normedweight))))**power
    
    return dCorr

In [ ]:
class BCEWithDecorrelationLoss(nn.Module):
    def __init__(self, lambda_corr=1.0):
        super().__init__()
        self.bce = nn.BCELoss()
        self.lambda_corr = lambda_corr

    def forward(self, y_pred, y_true, feature):
        bce_loss = self.bce(y_pred, y_true)
        corr_loss = distance_corr(y_pred, feature, torch.ones(len(y_pred)).to(device))
        total_loss = bce_loss + self.lambda_corr * corr_loss
        return total_loss
    
    def getBCE(self, y_pred, y_true):
        with torch.no_grad():
            return self.bce(y_pred, y_true).detach().cpu()
        
    def getDisCo(self, y_pred, feature):
        with torch.no_grad():
            return distance_corr(y_pred, feature, torch.ones(len(y_pred)).to(device)).detach().cpu()

In [ ]:
## Normalization factors (30 to 70 GeV)
#x_sec = [0.1117, 0.06028, 0.03098, 0.01537, 0.007179, 0.003207, 0.001322, 0.0005871, 0.0003560]
xsec_pos = [1.360E-01, 7.229E-02, 3.737E-02, 1.850E-02, 8.626E-03, 3.869E-03, 1.738E-03, 8.205E-04, 5.049E-04]
xsec_neg = [1.130E-01, 6.055E-02, 3.135E-02, 1.533E-02, 7.180E-03, 3.125E-03, 1.331E-03, 5.982E-04, 3.527E-04]
x_sec = [sum(x) for x in zip(xsec_pos, xsec_neg)]

k_fact = 1.3
lumi = 138000

norm_fact = []
for Zp_mass in x_sec:
    norm_fact.append(Zp_mass*k_fact*lumi/(300000))
#print(norm_fact)

mass_list = [30,35,40,45,50,55,60,65,70]

In [ ]:
training_variables = ['worstsip3d','worstIso','Ht', 'Mt', 'Vt', 'met', 'm3l_pT', 'pTL1', 'pTL2', 'pTL3',#'dPhiOS1', #'dPhiOS2', 'massZ2',
                      #'tightIdL1', 'tightIdL2', 'tightIdL3', 
                      'worstdxy', 'mvaTTHL1', 'mvaTTHL2', 'mvaTTHL3', #'massZ2',
                      #'mvaIdL3','mvaIdL1', 'mvaIdL2', #'worstdz', #'dRM1', 'dRM2', 'massZ0',
                      'ZpMass'] 
spectator_variables = ['massZ1', 'massZ2', 'Event', 'Run', 'ZpMass', 'pTL4', 'm4l', 'medIdL4', 'IsoL4', 'sip3dL4', 'm3l']
eval_variables = ['worstsip3d','worstIso','Ht', 'Mt', 'Vt', 'met', 'm3l_pT', 'pTL1', 'pTL2', 'pTL3',#'dPhiOS1', #'dPhiOS2', 'massZ2',
                  #'tightIdL1', 'tightIdL2', 'tightIdL3'
                  'worstdxy', 'mvaTTHL1', 'mvaTTHL2', 'mvaTTHL3', #'massZ2',
                  #'mvaIdL3', 'mvaIdL1', 'mvaIdL2',#'worstdz', 'dRM1', 'dRM2', 'massZ0',
                 ]
mass_var = ['massZ1']

In [ ]:
## New ntuples
filename_sgn = 'NtupleSkimmed/Wto3l_M1-70_v3fourMore_Jan25_noDiMuVeto.root' # same file as low mass
filename_sgn_ext = 'NtupleSkimmed/Wto3l_M1-70_v3fourMore_Jan25_noDiMuVeto_SRext.root' # with 20sigma SB
filename_bkg = 'NtupleSkimmed/Muon_Run2_noDuplicates_v3fourMore_Jan25_highMass_True_extension_False_noDiMuVeto.root'
path_bkgMC = 'NtupleSkimmed/FullRun_bkg_v3fourMore_Jan25_highMass_True_extension_False_noDiMuVeto/'

In [ ]:
# load data 
input_file_sgn = uproot.open(filename_sgn)
input_tree_sgn = get_arrays(input_file_sgn['passedEvents'], training_variables+spectator_variables)
input_file_bkg = uproot.open(filename_bkg)
input_tree_bkg = get_arrays(input_file_bkg['passedEvents'], training_variables+spectator_variables)

# prepare bkg and signal samples for training
preSel_bkg = 'ZpMass>25'#0
preSel_sgn = 'ZpMass>25'#0
#preSel_bkg = 'ZpMass==45 | ZpMass==50 | ZpMass==55 | ZpMass==65 | ZpMass==70 | ZpMass==75'
#preSel_sgn = 'ZpMass==45 | ZpMass==50 | ZpMass==55 | ZpMass==65 | ZpMass==70 | ZpMass==75'

bkg_trainX = get_input_features(input_tree_bkg, training_variables, preSel_bkg)
sig_trainX = get_input_features(input_tree_sgn, training_variables, preSel_sgn)
massBkg_trainX = get_input_features(input_tree_bkg, mass_var, preSel_bkg)
massSig_trainX = get_input_features(input_tree_sgn, mass_var, preSel_sgn)

print("bkg evts: ", len(bkg_trainX))
print("sig evts: ", len(sig_trainX))

In [ ]:
## Plotting input variables

it = 0
for inputVar in (training_variables):
    maxN = np.max(bkg_trainX[:,it])
    if (inputVar == 'worstdz' or inputVar == 'worstdxy'):
        plt.hist(bkg_trainX[:,it], label='bkg_train', density=True, alpha=0.5, bins=np.linspace(0, 0.1, 50))
        plt.hist(sig_trainX[:,it], label='sig_train', density=True, alpha=0.5, bins=np.linspace(0, 0.1, 50))
    else:
        plt.hist(bkg_trainX[:,it], label='bkg_train', density=True, alpha=0.5, bins=np.linspace(0, maxN, 50))
        plt.hist(sig_trainX[:,it], label='sig_train', density=True, alpha=0.5, bins=np.linspace(0, maxN, 50))
    
    plt.xlabel(inputVar)
    #plt.yscale("log")
    plt.legend()
    plt.show()
    it+=1

In [ ]:
#label signal and bkg & weight properly each mass point
bkg_trainY = np.zeros(len(bkg_trainX))
sig_trainY = np.ones(len(sig_trainX))
weights_bkg_trainY = np.zeros(len(bkg_trainX))
weights_sig_trainY = np.zeros(len(sig_trainX))
#weights_bkg_trainY = np.array([1./len(bkg_trainY)]*len(bkg_trainY))
#weights_sig_trainY = np.array([1./len(sig_trainY)]*len(sig_trainY))

n = 0
ZpMass_weight_bkg = np.zeros(len(mass_list)) 
ZpMass_weight_sig = np.zeros(len(mass_list)) 
for it in mass_list:
    evts_bkg = np.count_nonzero((bkg_trainX[:,(len(training_variables)-1)]) == it)
    evts_sig = np.count_nonzero((sig_trainX[:,(len(training_variables)-1)]) == it)
    ZpMass_weight_bkg[n] = 1./evts_bkg*len(mass_list)
    ZpMass_weight_sig[n] = 1./evts_sig*len(mass_list)
    n +=1

weights_bkg_trainY = np.zeros(len(bkg_trainY))
weights_sig_trainY = np.zeros(len(sig_trainY))
for i in range(len(bkg_trainY)):
    index = mass_list.index(bkg_trainX[i,(len(training_variables)-1)])
    weights_bkg_trainY[i] = ZpMass_weight_bkg[index]      
for i in range(len(sig_trainY)):
    index = mass_list.index(sig_trainX[i,(len(training_variables)-1)])
    weights_sig_trainY[i] = ZpMass_weight_sig[index]  

# split data into X and y
X = np.concatenate((bkg_trainX, sig_trainX))
Y = np.concatenate((bkg_trainY, sig_trainY))
W = np.concatenate((weights_bkg_trainY, weights_sig_trainY))

Mass = np.concatenate((massBkg_trainX, massSig_trainX))


In [ ]:
X_train, X_test = train_test_split(X, test_size=0.5, random_state=23)
Y_train, Y_test = train_test_split(Y, test_size=0.5, random_state=23)
W_train, W_test = train_test_split(W, test_size=0.5, random_state=23)
#print(X_train)

Mass_train, Mass_test = train_test_split(Mass, test_size=0.5, random_state=23)

In [ ]:
it = 0
for inputVar in (training_variables):
    maxN_train = np.max(X_train[:,it])
    #maxN_test = np.max(X_test[:,it])
    if (inputVar == 'worstdz' or inputVar == 'worstdxy'):
        plt.hist(bkg_trainX[:,it], label='bkg_train', density=True, alpha=0.5, bins=np.linspace(0, 0.1, 50))
        plt.hist(sig_trainX[:,it], label='sig_train', density=True, alpha=0.5, bins=np.linspace(0, 0.1, 50))
    else:
        plt.hist((X_train[:, it])[Y_train == 0], label='bkg_train', density=False, weights=(W_train[Y_train == 0]), alpha=0.5, bins=np.linspace(0, maxN_train, 50))
        plt.hist((X_train[:, it])[Y_train == 1], label='sgn_train', density=False, weights=(W_train[Y_train == 1]), alpha=0.5, bins=np.linspace(0, maxN_train, 50))
        #plt.hist((X_test[:, it])[Y_test == 0], label='bkg_test', density=False, weights=(W_test[Y_test == 0]), alpha=0.3, bins=np.linspace(0, maxN_test, 50))
        #plt.hist((X_test[:, it])[Y_test == 1], label='sgn_test', density=False, weights=(W_test[Y_test == 1]), alpha=0.3, bins=np.linspace(0, maxN_test, 50))
    plt.xlabel(inputVar)
    plt.legend()
    plt.show()
    it+=1

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train, sample_weight=W_train)
X_test = scaler.transform(X_test)

In [ ]:
X_train = torch.tensor(X_train, dtype=torch.float32)
Y_train = torch.tensor(Y_train, dtype=torch.float32).reshape(-1,1)
Mass_train = torch.tensor(Mass_train, dtype=torch.float32)
Mass_test = torch.tensor(Mass_test, dtype=torch.float32)

X_test = torch.tensor(X_test, dtype=torch.float32)
Y_test = torch.tensor(Y_test, dtype=torch.float32).reshape(-1,1)

In [ ]:
sampler = torch.utils.data.WeightedRandomSampler(W_train, len(W_train), replacement=True)

In [ ]:
class DataSet(torch.utils.data.Dataset):
    '''
    Class to define the dataset for the DataLoader.
    '''
    def __init__(self, data, labels, mass):
        self.data = data
        #self.constraint_obs = constraint_obs
        self.labels = labels
        self.mass = mass
        #self.xs_weights = xs_weights
        #self.weights = weights # Needed to calculate closure correctly

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        #return self.data[index], self.constraint_obs[index], self.labels[index], self.xs_weights[index], self.weights[index]
        return self.data[index], self.labels[index], self.mass[index]

In [ ]:
loader = torch.utils.data.DataLoader(DataSet(X_train, Y_train, Mass_train), batch_size=4096, sampler=sampler)

In [ ]:
# Fully connected NN with 3 layers
##The model expects rows of data with 7 variables (the first argument at the first layer set to 7)
##The first hidden layer has 12 neurons, followed by a ReLU activation function
##The second hidden layer has 7 neurons, followed by another ReLU activation function
##The output layer has one neuron, followed by a sigmoid activation function

if torch.cuda.is_available():         
    device = torch.device('cuda')
    print("Cuda available")
else:
    device = torch.device('cpu')
    print("Cuda NOT available, using cpu")


model = nn.Sequential(
    nn.Linear(len(training_variables), 2*len(training_variables)), ##11->15, 15, 20
    nn.ReLU(),
    nn.Linear(2*len(training_variables), len(training_variables)),
    nn.ReLU(),
    nn.Linear(len(training_variables), 1),
    nn.Sigmoid()
).to(device)
    
print(model)

#class MyClassifier(nn.Module):
#    def __init__(self):
#        super().__init__()
#        self.hidden1 = nn.Linear(8, 12)
#        self.act1 = nn.ReLU()
#        self.hidden2 = nn.Linear(12, 8)
#        self.act2 = nn.ReLU()
#        self.output = nn.Linear(8, 1)
#        self.act_output = nn.Sigmoid()
 
#    def forward(self, x):
#        x = self.act1(self.hidden1(x))
#        x = self.act2(self.hidden2(x))
#        x = self.act_output(self.output(x))
#        return x
 
#model = MyClassifier()
#print(model)

In [ ]:
#loss_fn = nn.BCELoss()  # binary cross entropy
loss_fn = BCEWithDecorrelationLoss(lambda_corr=38)
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
### Training
n_epochs = 200 # Epoch: Passes the entire training dataset to the model once
batch_size = 4096 # Batch: One or more samples passed to the model, from which the gradient descent algorithm will be executed for one iteration
model.train()

# some quantities to plot
train_losses = []
test_losses = []
train_BCE = []
train_DisCo = []
train_losses_perEpoch = []
y_pred_perEpoch = []
accuracy_train = []
accuracy_test = []

for epoch in range(n_epochs):
    for x,y,m in tqdm(loader, desc=f"Training epoch {epoch}", leave=False):
        y_pred = model(x.to(device))
        #loss = loss_fn(y_pred, y.to(device))
        loss = loss_fn(y_pred, y.to(device), m.to(device))
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        # remember the loss value at this step
        train_losses.append(loss.detach().item())
        train_BCE.append(loss_fn.getBCE(y_pred, y.to(device)))
        train_DisCo.append(loss_fn.getDisCo(y_pred, m.to(device)))
    #print(f'Finished epoch {epoch}, latest loss {loss.detach():.2f}')
    
    # evaluate test loss and metrics
    with torch.no_grad(): 
        y_pred_test = model(X_test.to(device))
        y_pred_train = model(X_train.to(device)) 
        test_losses.append( (loss_fn(y_pred_test[:2048], Y_test[:2048].to(device), Mass_test[:2048].to(device))).cpu())
        train_losses_perEpoch.append(loss.item())
        accuracy_train.append( (y_pred_train.round() == Y_train.to(device)).cpu().float().mean() )#
        accuracy_test.append( (y_pred_test.round() == Y_test.to(device)).cpu().float().mean() )#
    #print(f'Finished epoch {epoch}, latest loss {loss.detach():.2f}, accuracy {accuracy_test[-1]:.2f}')
 
plt.hist(y_pred_train[Y_train == 0].cpu(), label='bkg_train', density=True, alpha=0.5)
plt.hist(y_pred_train[Y_train == 1].cpu(), label='sig_train', density=True, alpha=0.5)
plt.hist(y_pred_test[Y_test == 0].cpu(), label='bkg_test', histtype='step', density=True, alpha=0.5)
plt.hist(y_pred_test[Y_test == 1].cpu(), label='sig_test', histtype='step', density=True, alpha=0.5)
plt.legend()

# Plotting performance
clear_output(wait=True)
plt.figure(figsize=(15, 3), dpi=100)
plt.subplot(1, 3, 1)
plt.plot(train_losses, label='train')
plt.plot(
  np.linspace(0, len(train_losses), len(test_losses) + 1)[1:],
  test_losses, label='test'
)
plt.ylabel("Loss")
plt.xlabel("# steps")
plt.legend()
    
plt.subplot(1, 3, 2)
plt.plot(train_losses_perEpoch, label='train')
plt.plot(test_losses, label='test')
plt.ylabel("Loss")
plt.xlabel("# epochs")
plt.ylim(0.28, 0.5)
plt.legend()
        
plt.subplot(1, 3, 3)
plt.plot(accuracy_train, label='train')
plt.plot(accuracy_test, label='test')
plt.ylabel("Accuracy")
plt.xlabel("# epochs")
plt.ylim(0.65, 0.9)
plt.legend()
plt.show()

In [ ]:
plt.subplot(1, 2, 1)
plt.plot(train_DisCo, label='train')
plt.ylabel("Loss DisCo")
plt.xlabel("# epochs")
plt.ylim(0, 0.05)
plt.xscale("log")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_BCE, label='train')
plt.ylabel("Loss BCE")
plt.xlabel("# epochs")
#plt.ylim(0, 0.05)
plt.legend()

In [ ]:
### Shapley values computation
torch.set_grad_enabled(False)
n_data = 100

# Define function to wrap model to transform data to tensor
f = lambda x: model( Variable((torch.from_numpy(x))).to(device)).cpu().detach().numpy()

# Convert my tensor to numpy
data = (X_test).cpu().numpy()
data_train = (X_train).cpu().numpy()
#data_sampled = shap.sample(data, n_data)
data_sampled = shap.sample(data, n_data)

# The explainer doesn't like tensors, hence the f function
explainer = shap.KernelExplainer(f, shap.kmeans(data, 100))#, feature_names=training_variables)

# Get the shap values from my test data
shap_values = explainer.shap_values(data_sampled)

# Enable the plots in jupyter
shap.initjs()

In [ ]:
means = np.abs(shap_values).mean(0).round(3)
mean_shap = np.zeros(len(means))

it = 0
for a in means:
    mean_shap[it] = a
    it += 1

fig, ax = plt.subplots()
ax.bar(training_variables, mean_shap)
print(mean_shap)

In [ ]:
shap_val_transp = np.zeros( (n_data,len(training_variables)) ) 
for n_row in range (0,n_data,1):
    for val in range (0,len(training_variables),1):
        shap_val_transp[n_row][val] = shap_values[n_row][val]

In [ ]:
feature_names = [ a + ": " + str(b) for a,b in zip(training_variables, mean_shap) ]
shap.summary_plot(shap_val_transp, data_sampled, max_display=data_sampled.shape[1], feature_names=feature_names)
print(feature_names)

In [ ]:
## NEW part

n_it = 1
bkg_evalY = []
sig_evalY = []
bkg_evalX = []
sgn_evalX = []

# prepare bkg and signal samples for evaluation of individual mass points
for n in mass_list:
    
    input_file_sgn = uproot.open(filename_sgn_ext)
    input_tree_sgn = get_arrays(input_file_sgn['passedEvents'], training_variables+spectator_variables)
    input_file_bkg = uproot.open(filename_bkg)
    input_tree_bkg = get_arrays(input_file_bkg['passedEvents'], training_variables+spectator_variables)
    
    phiVeto = '(massZ1<0.96 | massZ1>1.045) & (massZ2<0.96 | massZ2>1.045)'
    JPsiVeto = '(massZ1<2.9 | massZ1>3.25) & (massZ2<2.9 | massZ2>3.25)'
    PsiPrimeVeto = '(massZ1<3.62 | massZ1>3.75) & (massZ2<3.62 | massZ2>3.75)'
    UpsilonVeto = '(massZ1<9.0 | massZ1>11.0) & (massZ2<9.0 | massZ2>11.0)'
    veto_diMuReso = phiVeto +' & '+ JPsiVeto +' & '+ PsiPrimeVeto +' & '+ UpsilonVeto +' & '
    
    print("\n Mass point: ", n, " | iteration n.", n_it)
    #pre_sel_bkg_eval = veto_diMuReso + '(massZ1>'+str(n-(n*0.2))+' & massZ1<'+str(n-(n*0.02))+') | (massZ1>'+str(n+(n*0.02))+' & massZ1<'+str(n+(n*0.2))+')'
    pre_sel_bkg_eval = 'massZ1>0' #placeholder
    print("pre_sel_bkg_eval: ", pre_sel_bkg_eval)
    preSel_bkg_eval = pre_sel_bkg_eval #'(massZ1>48. & massZ1<58.8) | (massZ1>61.2 & massZ1<72.)' #'ZpMass>-1'
    preSel_sgn_eval = 'ZpMass=='+str(n)
    #preSel_sgn_eval = veto_diMuReso+'ZpMass=='+str(n) #ZpMass==60' # 'ZpMass==55 | ZpMass==65'

    bkg_evalX = get_input_features(input_tree_bkg, eval_variables, preSel_bkg_eval)
    sgn_evalX = get_input_features(input_tree_sgn, eval_variables, preSel_sgn_eval)
    bkg_evalY = np.zeros(len(bkg_evalX))
    sig_evalY = np.ones(len(sgn_evalX))
    print("bkg eval evts: ", len(bkg_evalX))
    print("sgn eval evts: ", len(sgn_evalX))
    print("bkg shape: ", bkg_evalX.shape)
    print("sgn shape: ", sgn_evalX.shape)
    
    test_bkg = np.full((len(bkg_evalX), 1), n)
    bkg_evalX = np.append(bkg_evalX,test_bkg,1)
    test_sgn = np.full((len(sgn_evalX), 1), n)
    sgn_evalX = np.append(sgn_evalX,test_sgn,1)
    #print(sgn_evalX)
    #print(bkg_evalX)
    
    X_eval_new = np.concatenate((bkg_evalX, sgn_evalX))
    Y_eval_new = np.concatenate((bkg_evalY, sig_evalY))
    print("X_eval_new: ", len(X_eval_new))
    print("Y_eval_new: ", len(X_eval_new))
    
    X_eval_new = scaler.transform(X_eval_new)
    Y_pred_new = model(torch.tensor(X_eval_new, dtype=torch.float32).to(device))
    print("X_eval_new transformed: ", len(X_eval_new))
    print("Y_pred_new: ", len(Y_pred_new))

    print(len(Y_pred_new[Y_eval_new == 0]))
    print(len(Y_pred_new[Y_eval_new == 1]))
    
    input_tree_bkg = input_tree_bkg[input_tree_bkg.eval(preSel_bkg_eval)]
    input_tree_sgn = input_tree_sgn[input_tree_sgn.eval(preSel_sgn_eval)]
    input_tree_bkg['DNN_score'] = (Y_pred_new[Y_eval_new == 0].cpu()).detach().numpy()
    input_tree_sgn['DNN_score'] = (Y_pred_new[Y_eval_new == 1].cpu()).detach().numpy()
    
    plt.hist(Y_pred_new[Y_eval_new == 0].detach().cpu().numpy(), label='bkg_train_'+str(n), density=True, alpha=0.5, bins=np.linspace(0, 1, 50))
    plt.hist(Y_pred_new[Y_eval_new == 1].detach().cpu().numpy(), label='sig_train_'+str(n), density=True, alpha=0.5, bins=np.linspace(0, 1, 50))
    plt.xlabel('pDNN_score (eval '+str(n)+')')
    plt.ylabel('Norm. entries')
    plt.legend()
    plt.show()

    ### Saving trees
    input_tree_bkg.to_csv("MiniTrees/MiniTree_bkg_pDNN_"+str(n)+"_new.csv")
    input_tree_bkg_RDF = ROOT.RDF.FromCSV("MiniTrees/MiniTree_bkg_pDNN_"+str(n)+"_new.csv", True, ',', 1000)
    print("Going to save root file bkg")
    input_tree_bkg_RDF.Snapshot("passedEvents", "MiniTrees/MiniTree_bkg_pDNN_"+str(n)+"_lambda38_new.root")

    input_tree_sgn.to_csv("MiniTrees/MiniTree_sgn_pDNN_"+str(n)+"_new.csv")
    input_tree_sgn_RDF = ROOT.RDF.FromCSV("MiniTrees/MiniTree_sgn_pDNN_"+str(n)+"_new.csv", True, ',', 1000)
    print("Going to save root file sgn")
    input_tree_sgn_RDF.Snapshot("passedEvents", "MiniTrees/MiniTree_sgn_pDNN_"+str(n)+"_lambda38_new.root")

    print("Done for mass ", n)
    n_it += 1
    

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

phiVeto = '(massZ1<0.96 | massZ1>1.045) & (massZ2<0.96 | massZ2>1.045)'
JPsiVeto = '(massZ1<2.9 | massZ1>3.25) & (massZ2<2.9 | massZ2>3.25)'
PsiPrimeVeto = '(massZ1<3.62 | massZ1>3.75) & (massZ2<3.62 | massZ2>3.75)'
UpsilonVeto = '(massZ1<9.0 | massZ1>11.0) & (massZ2<9.0 | massZ2>11.0)'
veto_diMuReso = phiVeto +' & '+ JPsiVeto +' & '+ PsiPrimeVeto +' & '+ UpsilonVeto +' & ' 

# load data (30-70 GeV)
var = ['DNN_score', 'massZ1', 'massZ2', 'ZpMass', 'pTL4']
bkg_fact = [0.113, 0.114, 0.114, 0.111, 0.106, 0.101, 0.102, 0.103, 0.099]

n_it = 0
for n in mass_list:
    #input_file_sgn_mini = uproot.open("/eos/user/c/caruta/Zp_FinalTrees/Feb18/MiniTree_sgn_pDNN_"+str(n)+".root")
    input_file_sgn_mini = uproot.open("MiniTrees/MiniTree_sgn_pDNN_"+str(n)+"_lambda38_new.root")
    input_tree_sgn_mini = get_arrays(input_file_sgn_mini['passedEvents'], var)
    #input_file_bkg_mini = uproot.open("/eos/user/c/caruta/Zp_FinalTrees/Feb18/MiniTree_bkg_pDNN_"+str(n)+".root")
    input_file_bkg_mini = uproot.open("MiniTrees/MiniTree_bkg_pDNN_"+str(n)+"_lambda38_new.root")
    input_tree_bkg_mini = get_arrays(input_file_bkg_mini['passedEvents'], var)

    print("\n Mass point: ", n, " | iteration n.", n_it)
    pre_sel_bkg_eval = veto_diMuReso + '((massZ1>'+str(n-(n*0.2))+' & massZ1<'+str(n-(n*0.02))+') | (massZ1>'+str(n+(n*0.02))+' & massZ1<'+str(n+(n*0.2))+')) & pTL4==0'
    print("pre_sel_bkg_eval: ", pre_sel_bkg_eval)
    preSel_bkg_eval = pre_sel_bkg_eval 
    preSel_sgn_eval = veto_diMuReso + 'ZpMass=='+str(n)+' & massZ1>'+str(n-(n*0.02))+' & massZ1<'+str(n+(n*0.02))+' & pTL4==0' 
    preSel_sgn_eval_20s = veto_diMuReso + 'ZpMass=='+str(n)+' & massZ1>'+str(n-(n*0.2))+' & massZ1<'+str(n+(n*0.2))+' & pTL4==0' 
    print("pre_sel_sgn_eval: ", preSel_sgn_eval)
    
    bkg_X = get_input_features(input_tree_bkg_mini, var, preSel_bkg_eval)
    sig_X = get_input_features(input_tree_sgn_mini, var, preSel_sgn_eval)
    sig_X_20s = get_input_features(input_tree_sgn_mini, var, preSel_sgn_eval_20s)
    bkg_Y = np.zeros(len(bkg_X))
    sig_Y = np.ones(len(sig_X))
    X = np.concatenate((bkg_X, sig_X))
    Y = np.concatenate((bkg_Y, sig_Y))
    print(f"bkg evts: {len(bkg_X)} ; in SR {len(bkg_X)*bkg_fact[n_it]:.2f}")
    print(f"sig evts: {len(sig_X)} ; normalized {len(sig_X)*norm_fact[n_it]:.2f}")
    print(f"sig evts 20s: {len(sig_X_20s)} ; normalized {len(sig_X_20s)*norm_fact[n_it]:.2f}; weight {norm_fact[n_it]:.4f}")

    plt.figure(figsize=(15, 3), dpi=100)
    plt.subplot(1, 3, 1) 
    plt.hist((bkg_X[:,0]), label='bkg_'+str(n), density=True, alpha=0.5, bins=np.linspace(0, 1, 30))
    plt.hist((sig_X[:,0]), label='sig_'+str(n), density=True, alpha=0.5, bins=np.linspace(0, 1, 30))
    plt.legend()
    #plt.show()

    auc = roc_auc_score(Y, X[:,0])
    fpr, tpr, thr = roc_curve(Y, X[:,0], drop_intermediate=True)

    plt.subplot(1, 3, 2) 
    plt.plot(fpr, tpr, label=f"AUC: {auc:.2f}")
    plt.ylabel("True positive rate")
    plt.xlabel("False positive rate")
    plt.legend()
    #plt.show()

    ### Std significance: S/sqrt(B) & AMS significance (signal normalized to 10^-4)
    sgn = np.array(tpr)*norm_fact[n_it]*len(sig_X)*0.01
    bkg = np.array(fpr)*bkg_fact[n_it]*len(bkg_X)
    sensitivity = np.zeros(len(fpr))
    AMS_sens = np.zeros(len(fpr))
    
    sensitivity = np.zeros(len(fpr))
    for i in range(len(fpr)):
        if ((fpr[i] != 0) & (tpr[i] != 0)):
            sensitivity[i] = np.array(sgn[i]/(np.sqrt(bkg[i])))
            AMS_sens[i] = np.sqrt( 2.* ( (sgn[i]+bkg[i]) * np.log(1+(sgn[i]/bkg[i])) - sgn[i] ) )
        else:
            sensitivity[i] = 0
            AMS_sens[i] = 0
    
    plt.subplot(1, 3, 3)
    plt.plot(thr, AMS_sens)
    plt.vlines(x=thr[AMS_sens.argmax()], ymin=min(AMS_sens), ymax=max(AMS_sens), color='red')
    plt.xlim(0.001, 1.0)
    plt.xlabel("pDNN threshold")
    plt.ylabel("Approx. Mean Significance")
    plt.show()
                  
    print(f"Max AMS sensitivity: {max(AMS_sens):.2f} for thr = {thr[AMS_sens.argmax()]:.4f}")
    bkg_rej_AMS = 1. - fpr[AMS_sens.argmax()]
    sgn_rej_AMS = 1. - tpr[AMS_sens.argmax()]
    print(f"bkg rejected: {bkg_rej_AMS*100:.2f}%")
    print(f"sgn rejected: {sgn_rej_AMS*100:.2f}%")
    print(f"Final bkg 20s: {(1-bkg_rej_AMS)*len(bkg_X):.2f}, in SR region: {(1-bkg_rej_AMS)*len(bkg_X)*bkg_fact[n_it]:.2f}")
    print(f"Final sgn in SR region: {(1-sgn_rej_AMS)*len(sig_X)*norm_fact[n_it]:.2f}")
    
    preSel_sgn_eval_20s_DNN = preSel_sgn_eval_20s+"& DNN_score>"+str(thr[AMS_sens.argmax()])
    sig_X_20s_DNN = get_input_features(input_tree_sgn_mini, var, preSel_sgn_eval_20s_DNN)
    print(f"Final sgn in 20s: {len(sig_X_20s_DNN)}; normalized {len(sig_X_20s_DNN)*norm_fact[n_it]:.2f}")

    print(f"Stats err bkg: {(1/np.sqrt((1-bkg_rej_AMS)*len(bkg_X))):.3f}")
    stats_err_AMS = 1.+(1/np.sqrt((1-bkg_rej_AMS)*len(bkg_X)))
    
    #plt.subplot(1, 4, 4)  
    plt.plot(thr, sensitivity)
    plt.vlines(x=thr[sensitivity.argmax()], ymin=min(sensitivity), ymax=max(sensitivity), color='red')
    plt.xlim(0.001, 1.0)
    plt.xlabel("DNN threshold")
    plt.show()

    print(f"Max sensitivity: {max(sensitivity):.2f} for thr = {thr[sensitivity.argmax()]:.4f}")
    bkg_rej = 1. - fpr[sensitivity.argmax()]
    sgn_rej = 1. - tpr[sensitivity.argmax()]

    print(f"bkg rejected: {bkg_rej*100:.2f}%")
    print(f"sgn rejected: {sgn_rej*100:.2f}%")
    print(f"Final bkg: {(1-bkg_rej)*len(bkg_X)*bkg_fact[n_it]:.2f}")
    print(f"Final sgn: {(1-sgn_rej)*len(sig_X)*norm_fact[n_it]:.2f}")
    print(f"Stats err bkg: {(1/np.sqrt((1-bkg_rej)*len(bkg_X))):.3f}")
    stats_err = 1.+(1/np.sqrt((1-bkg_rej)*len(bkg_X)))
    
    ### Write datacards (signal normalized to 10^-4)
    f = open("/eos/user/c/caruta/Wto3l_Combine/Combine_v8/CMSSW_10_2_13/src/HiggsAnalysis/CombinedLimit/NewDatacards/datacard_Zp"+str(n)+".txt", "w")
    f.write("imax 1  number of channels\njmax 1  number of backgrounds")
    f.write("\nkmax *  number of nuisance parameters (sources of systematical uncertainties)")
    f.write("\n------------\nbin bin1")
    f.write(f"\nobservation {(1-bkg_rej_AMS)*len(bkg_X)*bkg_fact[n_it]:.2f}")
    f.write("\n------------\nbin             bin1 bin1")
    f.write("\nprocess         Zp  dataSB\nprocess          0    1")
    f.write(f"\nrate         {(1-sgn_rej_AMS)*len(sig_X)*norm_fact[n_it]*0.01:.2f}"+f" {(1-bkg_rej_AMS)*len(bkg_X)*bkg_fact[n_it]:.2f}")
    f.close()
    
    f = open("/eos/user/c/caruta/Wto3l_Combine/Combine_v8/CMSSW_10_2_13/src/HiggsAnalysis/CombinedLimit/NewDatacards/datacard_Zp"+str(n)+".txt", "a")
    f.write("\n------------")
    f.write(f"\nstats  lnN    -    {stats_err_AMS:.3f}")
    f.close()
    n_it+=1

In [ ]:
### Fixed pDNN cut -- da sistemare!

from sklearn.metrics import roc_curve, roc_auc_score

phiVeto = '(massZ1<0.96 | massZ1>1.045) & (massZ2<0.96 | massZ2>1.045)'
JPsiVeto = '(massZ1<2.9 | massZ1>3.25) & (massZ2<2.9 | massZ2>3.25)'
PsiPrimeVeto = '(massZ1<3.62 | massZ1>3.75) & (massZ2<3.62 | massZ2>3.75)'
UpsilonVeto = '(massZ1<9.0 | massZ1>11.0) & (massZ2<9.0 | massZ2>11.0)'
veto_diMuReso = phiVeto +' & '+ JPsiVeto +' & '+ PsiPrimeVeto +' & '+ UpsilonVeto +' & ' 

cut_thr = 0.88
print(f"cut_thr: {cut_thr}")

# load data (30-70 GeV)
var = ['DNN_score', 'massZ1', 'massZ2', 'ZpMass', 'pTL4']
bkg_fact = [0.113, 0.114, 0.114, 0.111, 0.106, 0.101, 0.102, 0.103, 0.099]

n_it = 0
for n in mass_list:
    #input_file_sgn_mini = uproot.open("/eos/user/c/caruta/Zp_FinalTrees/Feb18/MiniTree_sgn_pDNN_"+str(n)+".root")
    input_file_sgn_mini = uproot.open("MiniTrees/MiniTree_sgn_pDNN_"+str(n)+"_lambda38_new.root")
    input_tree_sgn_mini = get_arrays(input_file_sgn_mini['passedEvents'], var)
    #input_file_bkg_mini = uproot.open("/eos/user/c/caruta/Zp_FinalTrees/Feb18/MiniTree_bkg_pDNN_"+str(n)+".root")
    input_file_bkg_mini = uproot.open("MiniTrees/MiniTree_bkg_pDNN_"+str(n)+"_lambda38_new.root")
    input_tree_bkg_mini = get_arrays(input_file_bkg_mini['passedEvents'], var)

    print("\n Mass point: ", n, " | iteration n.", n_it)
    pre_sel_bkg_eval = veto_diMuReso + '((massZ1>'+str(n-(n*0.2))+' & massZ1<'+str(n-(n*0.02))+') | (massZ1>'+str(n+(n*0.02))+' & massZ1<'+str(n+(n*0.2))+')) & pTL4==0'
    print("pre_sel_bkg_eval: ", pre_sel_bkg_eval)
    preSel_bkg_eval = pre_sel_bkg_eval +"& DNN_score>"+str(cut_thr)
    preSel_sgn_eval = veto_diMuReso + 'ZpMass=='+str(n)+' & massZ1>'+str(n-(n*0.02))+' & massZ1<'+str(n+(n*0.02))+' & pTL4==0'+"& DNN_score>"+str(cut_thr)
    preSel_sgn_eval_20s = veto_diMuReso + 'ZpMass=='+str(n)+' & massZ1>'+str(n-(n*0.2))+' & massZ1<'+str(n+(n*0.2))+' & pTL4==0' 
    print("pre_sel_sgn_eval: ", preSel_sgn_eval)
    
    bkg_X = get_input_features(input_tree_bkg_mini, var, preSel_bkg_eval)
    sig_X = get_input_features(input_tree_sgn_mini, var, preSel_sgn_eval)
    sig_X_20s = get_input_features(input_tree_sgn_mini, var, preSel_sgn_eval_20s)
    bkg_Y = np.zeros(len(bkg_X))
    sig_Y = np.ones(len(sig_X))
    X = np.concatenate((bkg_X, sig_X))
    Y = np.concatenate((bkg_Y, sig_Y))
    print(f"bkg evts: {len(bkg_X)} ; in SR {len(bkg_X)*bkg_fact[n_it]:.2f}")
    print(f"sig evts: {len(sig_X)} ; normalized {len(sig_X)*norm_fact[n_it]:.2f}")
    print(f"sig evts 20s: {len(sig_X_20s)} ; normalized {len(sig_X_20s)*norm_fact[n_it]:.2f}; weight {norm_fact[n_it]:.4f}")

    ## New part! 
    print(f"cut_thr: {cut_thr}")
    
    preSel_sgn_eval_20s_DNN = preSel_sgn_eval_20s+"& DNN_score>"+str(cut_thr)
    sig_X_20s_DNN = get_input_features(input_tree_sgn_mini, var, preSel_sgn_eval_20s_DNN)
    print(f"Final sgn in 20s: {len(sig_X_20s_DNN)}; normalized {len(sig_X_20s_DNN)*norm_fact[n_it]:.2f}")

    
    
    print(f"Stats err bkg: {(1/np.sqrt((1-bkg_rej_AMS)*len(bkg_X))):.3f}")
    stats_err_AMS = 1.+(1/np.sqrt((1-bkg_rej_AMS)*len(bkg_X)))


    print(f"bkg rejected: {bkg_rej*100:.2f}%")
    print(f"sgn rejected: {sgn_rej*100:.2f}%")
    print(f"Final bkg: {(1-bkg_rej)*len(bkg_X)*bkg_fact[n_it]:.2f}")
    print(f"Final sgn: {(1-sgn_rej)*len(sig_X)*norm_fact[n_it]:.2f}")
    print(f"Stats err bkg: {(1/np.sqrt((1-bkg_rej)*len(bkg_X))):.3f}")
    stats_err = 1.+(1/np.sqrt((1-bkg_rej)*len(bkg_X)))
    
    ### Write datacards (signal normalized to 10^-4)
    f = open("/eos/user/c/caruta/Wto3l_Combine/Combine_v8/CMSSW_10_2_13/src/HiggsAnalysis/CombinedLimit/NewDatacards/datacard_Zp"+str(n)+".txt", "w")
    f.write("imax 1  number of channels\njmax 1  number of backgrounds")
    f.write("\nkmax *  number of nuisance parameters (sources of systematical uncertainties)")
    f.write("\n------------\nbin bin1")
    f.write(f"\nobservation {(1-bkg_rej_AMS)*len(bkg_X)*bkg_fact[n_it]:.2f}")
    f.write("\n------------\nbin             bin1 bin1")
    f.write("\nprocess         Zp  dataSB\nprocess          0    1")
    f.write(f"\nrate         {(1-sgn_rej_AMS)*len(sig_X)*norm_fact[n_it]*0.01:.2f}"+f" {(1-bkg_rej_AMS)*len(bkg_X)*bkg_fact[n_it]:.2f}")
    f.close()
    
    f = open("/eos/user/c/caruta/Wto3l_Combine/Combine_v8/CMSSW_10_2_13/src/HiggsAnalysis/CombinedLimit/NewDatacards/datacard_Zp"+str(n)+".txt", "a")
    f.write("\n------------")
    f.write(f"\nstats  lnN    -    {stats_err_AMS:.3f}")
    f.close()
    n_it+=1

In [ ]:
## Save model
# Save the state dictionary
#torch.save(model.state_dict(), "pdnn_model_highMass_Apr18_DisCo_lambda38.pth")
torch.save(model.state_dict(), "pdnn_model_highMass_Nov6_DisCo_lambda38.pth")

In [ ]:
#cf_matrix = confusion_matrix(Y_train.cpu(), y_pred_train.cpu().round())
#df_cm = pd.DataFrame(cf_matrix / np.sum(cf_matrix, axis=1)[:, None], index = [i for i in ['bkg', 'sgn']],
#                     columns = [i for i in ['bkg', 'sgn']])
#plt.figure(figsize = (12,7))
#sn.heatmap(df_cm, annot=True)
#plt.savefig('ConfusionMatrix.png')